# ELT Case: Snowflake and Python

Purchase orders say what goods should cost. Supplier invoices say what was actually billed.
This notebook builds a pipeline that puts those two numbers side by side, from five sources in
five different formats, and then answers eight questions about what it shows.

Everything is driven from Python, but the work happens **inside Snowflake** - the code is almost
entirely `cs.execute(...)`.

**How this works.** Fill this notebook in, then commit and push it. The pushed notebook is the
submission - there is nothing else to hand in.

Part 1 builds the pipeline, one step per section. Part 2 answers eight questions about what it
shows.

Comments in the code are enough for Part 1. Each Part 2 question has an empty markdown cell under
it for your own reading of the result - **that is optional**, but some of these numbers have a
story behind them and a sentence saying what you make of one is worth more than the number alone.

Two practical things. The notebook should run **top to bottom on a clean kernel** - if a cell only
works because of something you ran earlier and deleted, it will not work for whoever opens it
next. And **do not commit your Snowflake password**; a secret in a commit stays in the git history
even after you delete the line.


## Part 1 - Build the pipeline

### Step 1 - Connect to Snowflake

Do **not** hard-code your password. This notebook is going into version control, and a secret in
a commit stays in the history even after you delete the line. Set the three environment variables
before launching VS Code, or read them from a file you keep out of the repository.


In [127]:
import os
import glob
import re
import csv
import pathlib
import snowflake.connector
from dotenv import load_dotenv
import snowflake.connector

load_dotenv()
# connect to Snowflake and create a cursor

conn = snowflake.connector.connect(
    user=os.getenv("USERNAME"),
    password=os.getenv("PASSWORD"),
    account=os.getenv("ACCOUNT_STRING"),
)

cs = conn.cursor()
cs.execute("SELECT CURRENT_USER(), CURRENT_ACCOUNT(), CURRENT_WAREHOUSE()")
print(cs.fetchone())

('W8HAN', 'JTB70375', 'COMPUTE_WH')


### Step 2 - Create the Snowflake objects

A warehouse for compute, a database for the case, and two schemas: `STAGE` for the internal
stages and file formats, `CORE` for the tables and views we build from them. Separating the
landing area from the modelled tables keeps it obvious which objects are raw and which are
derived.


In [39]:
# insert code: create the warehouse, database and the STAGE and CORE schemas
cs.execute("CREATE WAREHOUSE IF NOT EXISTS my_warehouse")
cs.execute("CREATE DATABASE IF NOT EXISTS my_case_db")
cs.execute("CREATE SCHEMA IF NOT EXISTS my_case_db.STAGE")
cs.execute("CREATE SCHEMA IF NOT EXISTS my_case_db.CORE")
print("Step 2 completed successfully!")

Step 2 completed successfully!


### Step 3 - Load the 41 purchase order files

The files are at line-item level, one per month. Three things to get right:

- **Skip the header row.** `SKIP_HEADER = 1` in the file format, or 41 header rows become data.
- **Do the transformation in the `COPY INTO`.** Select the columns you want and cast them there,
  rather than loading everything as text and fixing it afterwards.
- **Automate the `PUT`.** Iterate with `glob`; stage into `year/month` folders. It makes no
  practical difference at this size, but it is the right habit for data that arrives over time.

Columns dropped as not useful: `Comments` and `InternalComments` are almost entirely NULL,
`LastEditedBy` / `LastEditedWhen` (and their `Right_` duplicates) are audit fields, and
`PackageTypeID` carries a single value.

One data note: a few rows carry the date `2/29/2022`, which does not exist - 2022 was not a leap
year. `TRY_TO_DATE` returns NULL for those rather than failing the load, which is exactly why it
is used here instead of `TO_DATE`.


In [57]:
import glob

# Create file format - PARSE_HEADER lets MATCH_BY_COLUMN_NAME map columns automatically
cs.execute("""
CREATE OR REPLACE FILE FORMAT my_case_db.STAGE.my_csv_format
    TYPE = 'CSV'
    FIELD_DELIMITER = ','
    PARSE_HEADER = TRUE
    NULL_IF = ('NULL', '')
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    DATE_FORMAT = 'MM/DD/YYYY'
""")

# Create stage
cs.execute("CREATE OR REPLACE STAGE my_case_db.STAGE.my_stage")

# Create purchases table matching the real CSV header (23 columns)
# LastEditedWhen / Right_LastEditedWhen stored as STRING, not TIMESTAMP:
# some source months have malformed values like '00:00.0' that would
# otherwise cause the entire row to be dropped on load.
cs.execute("""
CREATE OR REPLACE TABLE my_case_db.CORE.purchases (
    PurchaseOrderID INTEGER,
    SupplierID INTEGER,
    OrderDate DATE,
    DeliveryMethodID INTEGER,
    ContactPersonID INTEGER,
    ExpectedDeliveryDate DATE,
    SupplierReference STRING,
    IsOrderFinalized INTEGER,
    Comments STRING,
    InternalComments STRING,
    LastEditedBy INTEGER,
    LastEditedWhen STRING,
    PurchaseOrderLineID INTEGER,
    StockItemID INTEGER,
    OrderedOuters NUMBER(10,2),
    Description STRING,
    ReceivedOuters NUMBER(10,2),
    PackageTypeID INTEGER,
    ExpectedUnitPricePerOuter NUMBER(10,2),
    LastReceiptDate DATE,
    IsOrderLineFinalized INTEGER,
    Right_LastEditedBy INTEGER,
    Right_LastEditedWhen STRING
)
""")

# Match all monthly CSV files
local_file_path = "/Users/apple/rsm-msba/mgta464/rsm-mgta464-snowflake-rsm-w8han/data/Monthly PO Data/*.csv"
matched_files = glob.glob(local_file_path)
print(f"Matched {len(matched_files)} files")

# Upload all matched CSV files to the stage
for file_path in matched_files:
    cs.execute(f"PUT 'file://{file_path}' @my_case_db.STAGE.my_stage")

print("Upload completed!")

# Load into purchases table
sql_copy = """
COPY INTO my_case_db.CORE.purchases
FROM @my_case_db.STAGE.my_stage
FILE_FORMAT = (FORMAT_NAME = 'my_case_db.STAGE.my_csv_format')
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
ON_ERROR = 'CONTINUE'
"""

try:
    cs.execute(sql_copy)
    results = cs.fetchall()
    for row in results:
        print(row)
except Exception as e:
    print("FULL ERROR:")
    print(str(e))

# Verify row count
cs.execute("SELECT COUNT(*) FROM my_case_db.CORE.purchases")
print(f"purchases: {cs.fetchone()[0]}")


Matched 41 files
Upload completed!
('my_stage/2019-4.csv.gz', 'LOADED', 190, 190, 190, 0, None, None, None, None)
('my_stage/2019-12.csv.gz', 'LOADED', 180, 180, 180, 0, None, None, None, None)
('my_stage/2019-11.csv.gz', 'LOADED', 179, 179, 179, 0, None, None, None, None)
('my_stage/2020-10.csv.gz', 'LOADED', 227, 227, 227, 0, None, None, None, None)
('my_stage/2020-11.csv.gz', 'LOADED', 205, 205, 205, 0, None, None, None, None)
('my_stage/2019-3.csv.gz', 'LOADED', 173, 173, 173, 0, None, None, None, None)
('my_stage/2019-6.csv.gz', 'LOADED', 183, 183, 183, 0, None, None, None, None)
('my_stage/2020-6.csv.gz', 'LOADED', 200, 200, 200, 0, None, None, None, None)
('my_stage/2019-5.csv.gz', 'LOADED', 195, 195, 195, 0, None, None, None, None)
('my_stage/2020-5.csv.gz', 'LOADED', 198, 198, 198, 0, None, None, None, None)
('my_stage/2019-7.csv.gz', 'LOADED', 199, 199, 199, 0, None, None, None, None)
('my_stage/2020-3.csv.gz', 'LOADED', 180, 180, 180, 0, None, None, None, None)
('my_stage/20

### Step 4 - Purchase order totals

Roll the line items up to one row per order. `POAmount` is the sum of
`ReceivedOuters * ExpectedUnitPricePerOuter` - **received**, not ordered. The gap between those
two is the whole point of the case.

`OrderDate` and `SupplierID` come along for the ride because they are constant within an order
and every downstream step needs them.


In [58]:
# insert code: build CORE.PURCHASE_ORDER_TOTALS with POAmount

cs.execute("""
CREATE OR REPLACE VIEW my_case_db.CORE.PURCHASE_ORDER_TOTALS AS
SELECT
    PurchaseOrderID,
    OrderDate,
    SupplierID,
    SUM(ReceivedOuters * ExpectedUnitPricePerOuter) AS POAmount
FROM my_case_db.CORE.purchases
GROUP BY
    PurchaseOrderID,
    OrderDate,
    SupplierID
""")

cs.execute("SELECT COUNT(*) FROM my_case_db.CORE.PURCHASE_ORDER_TOTALS")
print(f"PURCHASE_ORDER_TOTALS: {cs.fetchone()[0]}")

cs.execute("SELECT * FROM my_case_db.CORE.PURCHASE_ORDER_TOTALS LIMIT 5")
for row in cs.fetchall():
    print(row)

PURCHASE_ORDER_TOTALS: 2067
(561, datetime.date(2019, 12, 3), 7, Decimal('94009.2000'))
(565, datetime.date(2019, 12, 5), 7, Decimal('95413.2000'))
(569, datetime.date(2019, 12, 9), 7, Decimal('97272.4000'))
(576, datetime.date(2019, 12, 13), 4, Decimal('407994.0000'))
(579, datetime.date(2019, 12, 16), 4, Decimal('410286.0000'))


### Step 5 - Load and shred the supplier transactions

The XML lands in a `VARIANT` column first, then `LATERAL FLATTEN` turns each `<row>` into a
Snowflake row and `XMLGET` pulls the elements out of it.

Look at what is in the file before deciding what to keep. It is not all invoices: rows with
`TransactionTypeID` 5 are supplier invoices and carry a `PurchaseOrderID`; rows with
`TransactionTypeID` 7 are payments, have no purchase order, and have a zero ex-tax amount. Both
belong in the table - the join in step 6 is what filters to invoices.


In [ ]:
# insert code: file format, stage, raw VARIANT table, and the shredded CORE.SUPPLIER_TRANSACTIONS
cs.execute("""
CREATE OR REPLACE TABLE my_case_db.CORE.supplier_transactions AS
SELECT
    XMLGET(VALUE, 'SupplierTransactionID'):"$"::STRING AS TransactionID,
    XMLGET(VALUE, 'TransactionTypeID'):"$"::INTEGER AS TransactionTypeID,
    NULLIF(XMLGET(VALUE, 'PurchaseOrderID'):"$"::STRING, '')::INTEGER AS PurchaseOrderID,
    XMLGET(VALUE, 'SupplierID'):"$"::INTEGER AS SupplierID,
    TRY_TO_DATE(XMLGET(VALUE, 'TransactionDate'):"$"::STRING) AS TransactionDate,
    NULLIF(XMLGET(VALUE, 'AmountExcludingTax'):"$"::STRING, '')::NUMBER(10,2) AS AmountExcludingTax,
    NULLIF(XMLGET(VALUE, 'TaxAmount'):"$"::STRING, '')::NUMBER(10,2) AS TaxAmount,
    NULLIF(XMLGET(VALUE, 'TransactionAmount'):"$"::STRING, '')::NUMBER(10,2) AS TotalAmount
FROM my_case_db.STAGE.raw_supplier_transactions,
LATERAL FLATTEN(INPUT => RAW_DATA:"$")
""")

cs.execute("SELECT COUNT(*) FROM my_case_db.CORE.supplier_transactions")
print(f"Total transactions: {cs.fetchone()[0]}")

# Sanity check: should show both TypeID 5 and 7 with real counts
cs.execute(
    "SELECT TransactionTypeID, COUNT(*) FROM my_case_db.CORE.supplier_transactions GROUP BY TransactionTypeID"
)
for row in cs.fetchall():
    print(row)

# How many rows have a NULL date (the bad-data rows we caught)
cs.execute(
    "SELECT COUNT(*) FROM my_case_db.CORE.supplier_transactions WHERE TransactionDate IS NULL"
)
print(f"Rows with unparseable TransactionDate: {cs.fetchone()[0]}")

# Spot check individual rows
cs.execute("SELECT * FROM my_case_db.CORE.supplier_transactions LIMIT 5")
for row in cs.fetchall():
    print(row)

Total transactions: 2438
(5, 2072)
(7, 366)
Rows with unparseable TransactionDate: 5
('134', 5, 1, 2, datetime.date(2019, 1, 2), Decimal('313.50'), Decimal('47.03'), Decimal('360.53'))
('169', 5, 2, 4, datetime.date(2019, 1, 2), Decimal('21732.00'), Decimal('3259.80'), Decimal('24991.80'))
('186', 5, 3, 5, datetime.date(2019, 1, 2), Decimal('2740.50'), Decimal('411.11'), Decimal('3151.61'))
('215', 5, 4, 7, datetime.date(2019, 1, 2), Decimal('42481.20'), Decimal('6372.19'), Decimal('48853.39'))
('224', 5, 5, 10, datetime.date(2019, 1, 2), Decimal('35067.50'), Decimal('5260.14'), Decimal('40327.64'))


### Step 6 - Join orders to invoices

Inner join, so only orders that were invoiced survive. `invoiced_vs_quoted` is
`AmountExcludingTax - POAmount`: positive means the supplier billed more than the value of what
arrived.

Watch the grain. The join is order to invoice, one row each - if you join to `CORE.PURCHASES`
instead of the totals table you get one row per **line item** and every difference is counted
several times over.

The case asks for a materialized view. Snowflake materialized views cannot contain joins, so a
table is the right substitute here, exactly as the instructions allow.


In [66]:
# insert code: create purchase_orders_and_invoices with the invoiced_vs_quoted field

cs.execute("""
CREATE OR REPLACE TABLE my_case_db.CORE.purchase_orders_and_invoices AS
SELECT
    po.PurchaseOrderID,
    po.OrderDate,
    po.SupplierID,
    po.POAmount,
    inv.TransactionID,
    inv.TransactionTypeID,
    inv.TransactionDate,
    inv.AmountExcludingTax,
    inv.TaxAmount,
    inv.TotalAmount,
    inv.AmountExcludingTax - po.POAmount AS invoiced_vs_quoted
FROM my_case_db.CORE.PURCHASE_ORDER_TOTALS AS po
INNER JOIN my_case_db.CORE.supplier_transactions AS inv
    ON po.PurchaseOrderID = inv.PurchaseOrderID
""")

cs.execute("SELECT COUNT(*) FROM my_case_db.CORE.purchase_orders_and_invoices")
print(f"Total joined rows: {cs.fetchone()[0]}")

cs.execute("SELECT * FROM my_case_db.CORE.purchase_orders_and_invoices LIMIT 5")
for row in cs.fetchall():
    print(row)

# Sanity check: confirm every joined row is TransactionTypeID = 5 (invoices only)
# Payments (TypeID 7) have no PurchaseOrderID, so the inner join should exclude them automatically
cs.execute(
    "SELECT DISTINCT TransactionTypeID FROM my_case_db.CORE.purchase_orders_and_invoices"
)
print("Distinct TransactionTypeID in joined table:", cs.fetchall())

Total joined rows: 2065
(559, datetime.date(2019, 12, 2), 7, Decimal('92599.6000'), '82737', 5, datetime.date(2019, 12, 3), Decimal('92599.60'), Decimal('13889.94'), Decimal('106489.54'), Decimal('0.0000'))
(561, datetime.date(2019, 12, 3), 7, Decimal('94009.2000'), '83183', 5, datetime.date(2019, 12, 4), Decimal('94009.20'), Decimal('14101.38'), Decimal('108110.58'), Decimal('0.0000'))
(568, datetime.date(2019, 12, 9), 4, Decimal('407460.0000'), '84605', 5, datetime.date(2019, 12, 10), Decimal('407460.00'), Decimal('61119.00'), Decimal('468579.00'), Decimal('0.0000'))
(575, datetime.date(2019, 12, 12), 7, Decimal('96406.0000'), '85190', 5, datetime.date(2019, 12, 13), Decimal('96406.00'), Decimal('14460.90'), Decimal('110866.90'), Decimal('0.0000'))
(576, datetime.date(2019, 12, 13), 4, Decimal('407994.0000'), '85554', 5, datetime.date(2019, 12, 16), Decimal('407994.00'), Decimal('61199.10'), Decimal('469193.10'), Decimal('0.0000'))
Distinct TransactionTypeID in joined table: [(5,)]


### Step 7 - Bring the supplier data across from Postgres

The data must not pass through Python. Postgres writes it to a file with `COPY ... TO STDOUT`,
and Snowflake picks the file up from a stage.

The `CREATE TABLE` is generated from the file itself rather than typed by hand - a function that
reads the header and samples the data to pick a type per column. That is reusable; a hand-written
`CREATE TABLE` is not.


In [11]:
%pip install psycopg2-binary


/Users/apple/rsm-msba/.rsm-msba/envs/nix-uv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import psycopg2


In [56]:
# Step 7 - Bring the supplier data across from Postgres

import os
import csv
import re
import psycopg2
import snowflake.connector


# Connect to Snowflake
conn = snowflake.connector.connect(
    user=os.getenv("USERNAME"),
    password=os.getenv("PASSWORD"),
    account=os.getenv("ACCOUNT_STRING"),
)

cs = conn.cursor()


def snowflake_type(values):
    """Pick a Snowflake data type for a column, given a sample of its values."""
    values = [v.strip() for v in values if v is not None and v.strip() != ""]

    if not values:
        return "STRING"

    if all(re.fullmatch(r"-?\d+", v) for v in values):
        return "INTEGER"

    if all(re.fullmatch(r"-?\d+(\.\d+)?", v) for v in values):
        return "NUMBER(38,10)"

    if all(re.fullmatch(r"\d{4}-\d{2}-\d{2}", v) for v in values):
        return "DATE"

    if all(v.upper() in ("TRUE", "FALSE") for v in values):
        return "BOOLEAN"

    return "STRING"


def create_table_sql(csv_path, table_name, sample_rows=200):
    """Read a csv header and sample its rows, and return a CREATE TABLE statement."""

    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)

        rows = []
        for row in reader:
            rows.append(row)
            if len(rows) >= sample_rows:
                break

    column_values = [[] for _ in header]

    for row in rows:
        for i, value in enumerate(row):
            column_values[i].append(value)

    columns = []

    for name, values in zip(header, column_values):
        name = name.strip().replace('"', '""')
        data_type = snowflake_type(values)
        columns.append(f'"{name}" {data_type}')

    return f"""CREATE OR REPLACE TABLE {table_name} (
    {",\n    ".join(columns)}
)"""


# Connect to Postgres
pg_conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database="WestCoastImporters",
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
)

pg_cur = pg_conn.cursor()


# Postgres writes supplier_case directly to a CSV file
csv_path = "/Users/apple/rsm-msba/mgta464/rsm-mgta464-snowflake-rsm-w8han/data/supplier_case.csv"

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    pg_cur.copy_expert(
        """
        COPY supplier_case TO STDOUT
        WITH CSV HEADER
        """,
        f,
    )

print("supplier_case exported to CSV")


# Generate CREATE TABLE automatically from CSV
create_sql = create_table_sql(csv_path, "my_case_db.CORE.supplier_case")

print(create_sql)

cs.execute(create_sql)

print("Snowflake table created")


# Upload CSV to Snowflake stage
cs.execute(
    f"""
    PUT 'file://{csv_path}'
    @my_case_db.STAGE.my_stage
    """
)

print("CSV uploaded to Snowflake stage")


# Load CSV from stage into Snowflake
cs.execute(
    """
    COPY INTO my_case_db.CORE.supplier_case
    FROM @my_case_db.STAGE.my_stage
    FILE_FORMAT = (
        FORMAT_NAME = 'my_case_db.STAGE.my_csv_format'
    )
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
    ON_ERROR = 'CONTINUE'
    """
)

print("supplier_case loaded into Snowflake")


# Verify row count
cs.execute("SELECT COUNT(*) FROM my_case_db.CORE.supplier_case")

print(f"supplier_case rows: {cs.fetchone()[0]}")


# Show sample rows
cs.execute("SELECT * FROM my_case_db.CORE.supplier_case LIMIT 5")

for row in cs.fetchall():
    print(row)


print("Step 7 completed successfully!")


supplier_case exported to CSV
CREATE OR REPLACE TABLE my_case_db.CORE.supplier_case (
    "supplierid" INTEGER,
    "suppliername" STRING,
    "suppliercategoryid" INTEGER,
    "primarycontactpersonid" INTEGER,
    "alternatecontactpersonid" INTEGER,
    "deliverymethodid" INTEGER,
    "postalcityid" INTEGER,
    "supplierreference" STRING,
    "bankaccountname" STRING,
    "bankaccountbranch" STRING,
    "bankaccountcode" INTEGER,
    "bankaccountnumber" INTEGER,
    "bankinternationalcode" INTEGER,
    "paymentdays" INTEGER,
    "internalcomments" STRING,
    "phonenumber" STRING,
    "faxnumber" STRING,
    "websiteurl" STRING,
    "deliveryaddressline1" STRING,
    "deliveryaddressline2" STRING,
    "deliverypostalcode" INTEGER,
    "deliverylocation" STRING,
    "postaladdressline1" STRING,
    "postaladdressline2" STRING,
    "postalpostalcode" INTEGER,
    "lasteditedby" INTEGER,
    "validfrom" STRING,
    "validto" STRING
)
Snowflake table created
CSV uploaded to Snowflake sta

### Step 8 - Weather

The Marketplace subscription is the one thing that cannot be driven from Python - do it once in
the Snowflake web interface (**Data Products → Marketplace → NOAA → Weather & Environment →
Get**). After that, everything is SQL again.

Then three pieces of work:

1. **Load the ZCTA file** so every zip code has coordinates. It is tab delimited with seven
   columns, and the coordinates are the last two - read the header before you write the `COPY`.
2. **Find the nearest station to each supplier zip code.** Snowflake has a built-in `HAVERSINE`
   function, so there is no need to write the trigonometry by hand. Filter the station index to a
   rough bounding box first: comparing eight zip codes against every weather station on earth is
   a lot of arithmetic to throw away.
3. **Build `supplier_zip_code_weather`** - zip code, date, daily high - and join it to the orders.

One trap in the supplier data: at least one `postalpostalcode` is stored with four characters
where the ZCTA file has five. Pad it before you join or that supplier silently disappears.


In [121]:
# Step 8.1 - Load ZCTA zip code / latitude / longitude data

cs.execute("""
CREATE OR REPLACE TABLE my_case_db.STAGE.ZCTA (
    GEOID VARCHAR,
    ALAND NUMBER,
    AWATER NUMBER,
    ALAND_SQMI FLOAT,
    AWATER_SQMI FLOAT,
    INTPTLAT FLOAT,
    INTPTLONG FLOAT
)
""")

cs.execute("""
COPY INTO my_case_db.STAGE.ZCTA
FROM @my_case_db.STAGE.MY_STAGE/2021_Gaz_zcta_national.txt.gz
FILE_FORMAT = (
    TYPE = 'CSV'
    FIELD_DELIMITER = '\t'
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    COMPRESSION = 'AUTO'
)
ON_ERROR = 'ABORT_STATEMENT'
""")


In [122]:
# Step 8.2 - Find the nearest weather station for each supplier ZIP code

cs.execute("""
CREATE OR REPLACE TABLE my_case_db.CORE.supplier_zip_nearest_station AS

WITH supplier_zips AS (
    SELECT DISTINCT
        LPAD("postalpostalcode"::STRING, 5, '0') AS zip_code
    FROM my_case_db.CORE.supplier_case
    WHERE "postalpostalcode" IS NOT NULL
),

supplier_locations AS (
    SELECT
        s.zip_code,
        z.INTPTLAT AS zip_lat,
        z.INTPTLONG AS zip_long
    FROM supplier_zips s
    INNER JOIN my_case_db.STAGE.ZCTA z
        ON s.zip_code = LPAD(z.GEOID::STRING, 5, '0')
),

temperature_stations AS (
    SELECT DISTINCT
        t.NOAA_WEATHER_STATION_ID AS station_id,
        i.LATITUDE,
        i.LONGITUDE
    FROM SNOWFLAKE_PUBLIC_DATA_CORE_WEATHER_DATA.PUBLIC_DATA.NOAA_WEATHER_METRICS_TIMESERIES t
    INNER JOIN SNOWFLAKE_PUBLIC_DATA_CORE_WEATHER_DATA.PUBLIC_DATA.NOAA_WEATHER_STATION_INDEX i
        ON t.NOAA_WEATHER_STATION_ID = i.NOAA_WEATHER_STATION_ID
    WHERE t.VARIABLE = 'maximum_temperature'
      AND i.LATITUDE IS NOT NULL
      AND i.LONGITUDE IS NOT NULL
),

station_distances AS (
    SELECT
        s.zip_code,
        t.station_id,
        HAVERSINE(
            s.zip_lat,
            s.zip_long,
            t.LATITUDE,
            t.LONGITUDE
        ) AS distance_km
    FROM supplier_locations s
    CROSS JOIN temperature_stations t
    WHERE t.LATITUDE BETWEEN s.zip_lat - 5 AND s.zip_lat + 5
      AND t.LONGITUDE BETWEEN s.zip_long - 5 AND s.zip_long + 5
)

SELECT
    zip_code,
    station_id,
    distance_km
FROM station_distances

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY zip_code
    ORDER BY distance_km
) = 1
""")


In [123]:
# Step 8.3 - Build supplier ZIP code weather table

cs.execute("""
CREATE OR REPLACE TABLE my_case_db.CORE.supplier_zip_code_weather AS

SELECT
    ns.zip_code,
    m.DATE AS weather_date,
    m.VALUE AS daily_high_temperature

FROM my_case_db.CORE.supplier_zip_nearest_station ns

INNER JOIN SNOWFLAKE_PUBLIC_DATA_CORE_WEATHER_DATA.PUBLIC_DATA.NOAA_WEATHER_METRICS_TIMESERIES m
    ON ns.station_id = m.NOAA_WEATHER_STATION_ID

WHERE m.VARIABLE = 'maximum_temperature'
  AND m.DATE IS NOT NULL
  AND m.VALUE IS NOT NULL
""")


In [124]:
# Step 8.4 - Final join: orders/invoices + suppliers + weather

cs.execute("""
CREATE OR REPLACE TABLE my_case_db.CORE.purchase_orders_invoices_weather AS

SELECT
    p.*,
    s."suppliername",
    LPAD(s."postalpostalcode"::STRING, 5, '0') AS zip_code,
    w.weather_date,
    w.daily_high_temperature

FROM my_case_db.CORE.purchase_orders_and_invoices AS p

INNER JOIN my_case_db.CORE.supplier_case AS s
    ON p.supplierid = s."supplierid"

INNER JOIN my_case_db.CORE.supplier_zip_code_weather AS w
    ON LPAD(s."postalpostalcode"::STRING, 5, '0') = w.zip_code
   AND p.transactiondate = w.weather_date
""")


## Part 2 - Questions

Each question has a single numeric answer. Run the query; the query and the number are what is
being marked.

Under each one there is a markdown cell for your own reading of the result. Filling it in is
optional - but try it where you see something worth saying.


### Question 1. Across every purchase order in the data, what is the total value of the goods that were **actually received**? Two decimals.


In [129]:
# insert code: answer the question above with a single query
cs.execute("""
SELECT ROUND(SUM(POAMOUNT), 2) AS total_value_actually_received
FROM my_case_db.CORE.purchase_orders_and_invoices
""")

print(cs.fetchone()[0])

935079013.60


_Optional - what do you make of this number?_


### Question 2. Not every row in the supplier transaction XML is an invoice. How many are **not** invoices against a purchase order?


In [ ]:
# insert code: answer the question above with a single query
cs.execute("""
SELECT COUNT(*) AS not_invoices_against_purchase_order
FROM my_case_db.CORE.SUPPLIER_TRANSACTIONS t
WHERE NOT EXISTS (
    SELECT 1
    FROM my_case_db.CORE.PURCHASE_ORDERS_AND_INVOICES p
    WHERE p.TRANSACTIONID = t.TRANSACTIONID
)
""")

print(cs.fetchone()[0])

373


_Optional - what do you make of this number?_


### Question 3. Across all purchase orders that were invoiced, what is the **total amount billed in excess** of the value of goods received?


In [135]:
# insert code: answer the question above with a single query
cs.execute("""
SELECT
    ROUND(SUM(
        CASE
            WHEN INVOICED_VS_QUOTED > 0
            THEN INVOICED_VS_QUOTED
            ELSE 0
        END
    ), 2) AS total_amount_billed_in_excess
FROM my_case_db.CORE.PURCHASE_ORDERS_AND_INVOICES
""")

print(cs.fetchone()[0])

2869980.00


_Optional - what do you make of this number?_


### Question 4. What share of the total received value comes from the **single largest supplier**? As a percentage, two decimals.


In [136]:
# insert code: answer the question above with a single query
cs.execute("""
WITH supplier_received AS (
    SELECT
        t.SUPPLIERID,
        SUM(t.AMOUNTEXCLUDINGTAX) AS received_value
    FROM my_case_db.CORE.SUPPLIER_TRANSACTIONS t
    WHERE t.PURCHASEORDERID IS NOT NULL
    GROUP BY t.SUPPLIERID
),

ranked_suppliers AS (
    SELECT
        SUPPLIERID,
        received_value,
        SUM(received_value) OVER () AS total_received_value,
        ROW_NUMBER() OVER (ORDER BY received_value DESC) AS rn
    FROM supplier_received
)

SELECT
    ROUND(
        100.0 * received_value / total_received_value,
        2
    ) AS largest_supplier_share_pct
FROM ranked_suppliers
WHERE rn = 1
""")

print(cs.fetchone()[0])

71.80


_Optional - what do you make of this number?_


### Question 5. Looking at the monthly total value of goods received, which month saw the **largest increase over the month before it**? Answer as `YYYYMM`.

This one needs a window function: the comparison is between a row and the row before it in time.


In [137]:
# insert code: answer the question above with a single query
cs.execute("""
WITH monthly_received AS (
    SELECT
        TO_CHAR(ORDERDATE, 'YYYYMM') AS yyyymm,
        SUM(POAMOUNT) AS total_received
    FROM my_case_db.CORE.PURCHASE_ORDERS_AND_INVOICES
    GROUP BY TO_CHAR(ORDERDATE, 'YYYYMM')
),

monthly_change AS (
    SELECT
        yyyymm,
        total_received,
        total_received
            - LAG(total_received) OVER (ORDER BY yyyymm) AS increase
    FROM monthly_received
)

SELECT
    yyyymm,
    increase
FROM monthly_change
WHERE increase IS NOT NULL
ORDER BY increase DESC
LIMIT 1
""")

rows = cs.fetchall()
print(rows)

[('202203', Decimal('11981402.8000'))]


_Optional - what do you make of this number?_


### Question 6. Between the first and last order date, how many **calendar days** went by with no purchase order at all?

You cannot count rows that are not there - the calendar has to be generated first, then the orders anti-joined against it.


In [138]:
# insert code: answer the question above with a single query
cs.execute("""
WITH order_dates AS (
    SELECT
        MIN(ORDERDATE) AS first_order_date,
        MAX(ORDERDATE) AS last_order_date
    FROM my_case_db.CORE.PURCHASE_ORDERS_AND_INVOICES
),

calendar AS (
    SELECT
        DATEADD(
            DAY,
            SEQ4(),
            first_order_date
        ) AS calendar_date
    FROM order_dates,
         TABLE(
             GENERATOR(
                 ROWCOUNT => 10000
             )
         )
    WHERE DATEADD(
              DAY,
              SEQ4(),
              first_order_date
          ) <= last_order_date
),

order_days AS (
    SELECT DISTINCT
        ORDERDATE::DATE AS order_date
    FROM my_case_db.CORE.PURCHASE_ORDERS_AND_INVOICES
)

SELECT
    COUNT(*) AS days_without_purchase_order
FROM calendar c
LEFT JOIN order_days o
    ON c.calendar_date = o.order_date
WHERE o.order_date IS NULL
""")

print(cs.fetchone()[0])

187


_Optional - what do you make of this number?_


### Question 7. Of the supplier zip codes in `supplier_case`, which one is **furthest north**?


In [142]:
# insert code: answer the question above with a single query
cs.execute("""
SELECT
    LPAD(s."postalpostalcode"::STRING, 5, '0') AS zip_code,
    z."INTPTLAT" AS latitude
FROM my_case_db.CORE.SUPPLIER_CASE AS s
INNER JOIN my_case_db.STAGE.ZCTA AS z
    ON LPAD(s."postalpostalcode"::STRING, 5, '0') = z."GEOID"
WHERE s."postalpostalcode" IS NOT NULL
ORDER BY z."INTPTLAT" DESC
LIMIT 1
""")

print(cs.fetchone())

('60523', 41.836395)


_Optional - what do you make of this number?_


### Question 8. How many suppliers in `supplier_case` **never appear on a purchase order**?


In [154]:
# insert code: answer the question above with a single query
cs.execute("""
SELECT COUNT(*)
FROM MY_CASE_DB.CORE.SUPPLIER_CASE
WHERE "supplierid" IS NOT NULL
  AND NOT EXISTS (
      SELECT 1
      FROM MY_CASE_DB.CORE.PURCHASE_ORDERS_AND_INVOICES
      WHERE PURCHASE_ORDERS_AND_INVOICES.SUPPLIERID
            = SUPPLIER_CASE."supplierid"
  )
""")

cs.fetchall()

[(6,)]

_Optional - what do you make of this number?_


## Close the connection


In [155]:
# insert code: close the cursor and the connection
cs.close()
conn.close()